# Capítulo 8 — Geração da resposta (Response Generation)

**Livro:** começa na página **400** do PDF.

**Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")  # ia.py (modelos via OpenRouter) e corpus.py (base de conhecimento de exemplo)
from ia import *
from corpus import DOCS
import re, json, math, random
import numpy as np
print("Pronto. Modelo padrão:", MODELO, "| documentos na base:", len(DOCS))

## Improving LLM output quality by refining prompts · PDF p. 401

#### Exercício: Resposta com citações

Monte o RAG completo: recupere os 3 documentos mais relevantes (BM25), coloque no *prompt* com `[id]` e instrua o modelo a responder **só** com base neles, citando os ids. Teste com uma pergunta respondível e uma que não está na base.

In [ ]:
from rank_bm25 import BM25Okapi
bm = BM25Okapi([re.findall(r"\w+", (d["titulo"] + " " + d["texto"]).lower()) for d in DOCS])
SISTEMA = "Você responde dúvidas da Biblioteca Aurora usando SÓ os trechos dados. Cite os ids entre colchetes. Se a resposta não estiver nos trechos, diga que não sabe."
def rag(q, k=3):
    top = np.argsort(-bm.get_scores(re.findall(r"\w+", q.lower())))[:k]
    ctx = "\n".join(f"[{DOCS[i]['id']}] {DOCS[i]['texto']}" for i in top)
    return perguntar(f"Trechos:\n{ctx}\n\nPergunta: {q}", sistema=SISTEMA, temperature=0)
print(rag("Quanto custa imprimir 10 páginas coloridas?")); print(rag("A biblioteca tem estacionamento?"))